# 09. SMART Barangs & Lots Fine-Grained Subcategory Consolidation
### Modular, Subcategory-by-Subcategory Deduplication & Master Data Synchronization
---
This notebook serves as the final refinement stage of the AMS-to-SMART data migration:
1. **Granular Normalization**: Iterates through individual subcategories to resolve messy specifications, embedded location strings, and syntax anomalies.
2. **Modular Architecture**: Each subcategory has its own dedicated, self-contained execution cell.
3. **Master Re-Synchronization**: After updating any subcategory, `sync_consolidated_masters()` automatically aggregates all 111 subcategories, re-indexes sequential IDs, re-links units, and re-exports all staging and final deliverables.
4. **Data Integrity Assured**: Every execution re-validates 100% of global referential and schema assertions.

In [1]:
import os
import sys
import json
import re
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Dynamic Path Resolution
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path('/home/jovyan/data'),                      # Docker container path
        Path('data'),                                  # Repository root
        Path('../data'),                               # Notebooks directory
        Path('/home/ran/Devs/ams-smart-migration/data') # Host absolute path
    ]
    for p in candidates:
        if p.exists() and (p / 'smart').exists():
            return p.resolve()
    return Path('data').resolve()

DATA_DIR = get_data_dir()
STAGING_ASSETS_DIR = DATA_DIR / 'smart' / 'staging' / 'assets'
STAGING_UNITS_DIR = DATA_DIR / 'smart' / 'staging' / 'units'
STAGING_BARANGS_DIR = DATA_DIR / 'smart' / 'staging' / 'barangs'
STAGING_LOTS_DIR = DATA_DIR / 'smart' / 'staging' / 'lots'

FINAL_BARANG_DIR = DATA_DIR / 'smart' / 'final' / 'barang'
FINAL_LOT_DIR = DATA_DIR / 'smart' / 'final' / 'lot'
FINAL_UNIT_DIR = DATA_DIR / 'smart' / 'final' / 'unit'

FINAL_SUBCATEGORY_PATH = DATA_DIR / 'smart' / 'final' / 'subcategory' / 'subcategories.csv'
FINAL_BRAND_PATH = DATA_DIR / 'smart' / 'final' / 'brand' / 'brands.csv'
FINAL_VENDOR_PATH = DATA_DIR / 'smart' / 'final' / 'vendor' / 'vendors.csv'

STAGING_BRAND_MAP_PATH = DATA_DIR / 'smart' / 'staging' / 'brand' / 'brand_ams_mapping.csv'
STAGING_VENDOR_MAP_PATH = DATA_DIR / 'smart' / 'staging' / 'vendor' / 'vendor_ams_mapping.csv'
UNITS_MANIFEST_PATH = STAGING_UNITS_DIR / '_units_manifest.json'

for d in [STAGING_BARANGS_DIR, STAGING_LOTS_DIR, FINAL_BARANG_DIR, FINAL_LOT_DIR, FINAL_UNIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. Load Master Lookups
# ---------------------------------------------------------
df_sub = pd.read_csv(FINAL_SUBCATEGORY_PATH, dtype=str)
sub_map = dict(zip(df_sub['code'], df_sub['id']))

df_brand = pd.read_csv(FINAL_BRAND_PATH, dtype=str)
brand_name_to_id = {r['name'].strip().upper(): str(r['id']).strip() for _, r in df_brand.iterrows() if pd.notna(r['name'])}

df_bmap = pd.read_csv(STAGING_BRAND_MAP_PATH, dtype=str)
ams_brand_to_id = {}
for _, r in df_bmap.iterrows():
    if pd.notna(r.get('ams_brand_name')):
        ams_brand_to_id[r['ams_brand_name'].strip().upper()] = str(r['smart_brand_id']).strip()

brand_aliases = {
    'LIONSTAR': brand_name_to_id.get('LION STAR', '125'),
    'LION STAR': brand_name_to_id.get('LION STAR', '125'),
    'PONGS': brand_name_to_id.get('PONGS HOME', '79'),
    'PONGS HOME': brand_name_to_id.get('PONGS HOME', '79'),
    'SYLYART': brand_name_to_id.get('SYLY ART', '130'),
    'SYLU ART': brand_name_to_id.get('SYLY ART', '130'),
    'SOFA': '123',
    'NO BRAND': '123',
    'CHITOSE': '173',
    'ICA': '174',
    'STELA': '175',
    'STELLA': '175',
    'MORELLY': '176',
}

df_vmap = pd.read_csv(STAGING_VENDOR_MAP_PATH, dtype=str)
vendor_map = {}
for _, r in df_vmap.iterrows():
    v_id = str(r['ams_id']).strip()
    if v_id.endswith('.0'): v_id = v_id[:-2]
    vendor_map[v_id] = str(r['smart_vendor_id']).strip()

def resolve_brand_id(raw_brand):
    if pd.isna(raw_brand) or not str(raw_brand).strip():
        return '123'
    b = str(raw_brand).strip().upper()
    if b in brand_aliases:
        return brand_aliases[b]
    if b in brand_name_to_id:
        return brand_name_to_id[b]
    if b in ams_brand_to_id:
        return ams_brand_to_id[b]
    return '123'

def resolve_vendor_id(raw_vendor):
    if pd.isna(raw_vendor) or str(raw_vendor).strip() in ('', 'NULL', 'null', 'None'):
        return None
    v = str(raw_vendor).strip()
    if v.endswith('.0'): v = v[:-2]
    return vendor_map.get(v, None)

def resolve_organizer_id(raw_org):
    return '2' if str(raw_org).strip().upper() == 'ICT' else '1'

def clean_text(s):
    if pd.isna(s) or s is None:
        return ''
    return re.sub(r'\s+', ' ', str(s).replace('\r', ' ').replace('\n', ' ')).strip()

# ---------------------------------------------------------
# 3. Master Re-Synchronization Engine
# ---------------------------------------------------------
def escape_sql(val):
    if val is None or pd.isna(val):
        return 'NULL'
    s = str(val).replace("'", "''")
    return f"N'{s}'"

def format_num_sql(val):
    if val is None or pd.isna(val) or str(val).strip() in ('', 'None', 'nan', 'NULL'):
        return 'NULL'
    try:
        f_val = float(val)
        return str(int(f_val)) if f_val.is_integer() else str(f_val)
    except (ValueError, TypeError):
        return str(val)

def sync_consolidated_masters():
    with open(UNITS_MANIFEST_PATH, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
        
    all_barangs = []
    all_lots = []
    all_units = []
    
    current_id = 1
    barangs_manifest = {'subcategories': {}, 'total_barangs': 0}
    lots_manifest = {'subcategories': {}, 'total_lots': 0}
    units_manifest = {'subcategories': {}, 'total_units': 0}
    
    for sub_code, meta in sorted(manifest['subcategories'].items()):
        b_file = STAGING_BARANGS_DIR / meta['file']
        l_file = STAGING_LOTS_DIR / meta['file']
        u_file = STAGING_UNITS_DIR / meta['file']
        
        if not b_file.exists() or meta['row_count'] == 0:
            continue
            
        df_b = pd.read_csv(b_file, dtype=str)
        df_l = pd.read_csv(l_file, dtype=str)
        df_u = pd.read_csv(u_file, dtype=str)
        
        # Re-index IDs sequentially for this partition
        old_b_to_new_id = {}
        for idx in range(len(df_b)):
            old_b_to_new_id[str(df_b.at[idx, 'id'])] = current_id
            df_b.at[idx, 'id'] = current_id
            df_l.at[idx, 'id'] = current_id
            df_l.at[idx, 'barang_id'] = current_id
            current_id += 1
            
        # Save re-indexed partition files
        df_b.to_csv(b_file, index=False)
        df_l.to_csv(l_file, index=False)
        
        # Update units lot_id mapping
        if 'lot_id' in df_u.columns:
            df_u['lot_id'] = df_u['lot_id'].map(lambda x: old_b_to_new_id.get(str(x), x))
            for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
                if c in df_u.columns:
                    df_u[c] = pd.to_numeric(df_u[c], errors='coerce').astype('Int64')
            df_u.to_csv(u_file, index=False)
            
        all_barangs.append(df_b)
        all_lots.append(df_l)
        all_units.append(df_u)
        
        barangs_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_b)}
        lots_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_l)}
        units_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_u)}
        
    df_barangs_master = pd.concat(all_barangs, ignore_index=True)
    df_lots_master = pd.concat(all_lots, ignore_index=True)
    df_units_master = pd.concat(all_units, ignore_index=True)
    
    barangs_manifest['total_barangs'] = len(df_barangs_master)
    lots_manifest['total_lots'] = len(df_lots_master)
    units_manifest['total_units'] = len(df_units_master)
    
    # 1. Staging Masters
    df_barangs_master.to_csv(STAGING_BARANGS_DIR / 'barangs.csv', index=False)
    with open(STAGING_BARANGS_DIR / '_barangs_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(barangs_manifest, f, indent=2)
        
    df_lots_master.to_csv(STAGING_LOTS_DIR / 'lots.csv', index=False)
    with open(STAGING_LOTS_DIR / '_lots_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(lots_manifest, f, indent=2)
        
    df_units_master.to_csv(STAGING_UNITS_DIR / 'units.csv', index=False)
    with open(STAGING_UNITS_DIR / '_units_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(units_manifest, f, indent=2)
        
    # 2. Final Production Masters
    df_barangs_master.to_csv(FINAL_BARANG_DIR / 'barangs.csv', index=False)
    df_lots_master.to_csv(FINAL_LOT_DIR / 'lots.csv', index=False)
    
    db_ordered_cols = [
        'id', 'number', 'legacy_number', 'lot_id', 'location_id',
        'status', 'condition', 'price', 'image_url', 'vehicle_registration',
        'burden', 'project_id', 'created_at', 'updated_at', 'type',
        'classification', 'vendor_id'
    ]
    df_units_final = df_units_master[[
        'id', 'number', 'legacy_number', 'lot_id', 'location_id',
        'status', 'condition', 'price', 'image_url', 'burden',
        'project_id', 'created_at', 'updated_at', 'type',
        'classification', 'vendor_id'
    ]].copy()
    df_units_final['vehicle_registration'] = None
    df_units_final = df_units_final[db_ordered_cols]
    for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
        df_units_final[c] = pd.to_numeric(df_units_final[c], errors='coerce').astype('Int64')
    df_units_final.to_csv(FINAL_UNIT_DIR / 'units.csv', index=False)
    
    # 3. SQL Insert Scripts
    with open(FINAL_BARANG_DIR / 'barangs.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [barangs] ON;\nGO\n\n')
        for _, r in df_barangs_master.iterrows():
            f.write(f"INSERT INTO [barangs] ([id], [number], [subcategory_id], [brand_id], [uom_id], [name], [specification], [min_stock_threshold], [image_url], [last_restock_at], [created_at], [updated_at]) VALUES ({r['id']}, {escape_sql(r['number'])}, {r['subcategory_id']}, {r['brand_id']}, {r['uom_id']}, {escape_sql(r['name'])}, {escape_sql(r['specification'])}, {format_num_sql(r['min_stock_threshold'])}, {escape_sql(r['image_url'])}, {escape_sql(r['last_restock_at'])}, '{r['created_at']}', '{r['updated_at']}');\n")
        f.write('\nGO\nSET IDENTITY_INSERT [barangs] OFF;\nGO\n')
        
    with open(FINAL_LOT_DIR / 'lots.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [lots] ON;\nGO\n\n')
        for _, r in df_lots_master.iterrows():
            f.write(f"INSERT INTO [lots] ([id], [number], [barang_id], [organizer_id], [vendor_id], [legacy_vendor_id], [location_id], [initial_quantity], [current_quantity], [po_number], [date_of_receipt], [unit_price], [image_url], [burden], [project_id], [created_at], [updated_at]) VALUES ({r['id']}, {escape_sql(r['number'])}, {r['barang_id']}, {r['organizer_id']}, NULL, NULL, NULL, NULL, NULL, NULL, '{r['date_of_receipt']}', NULL, {escape_sql(r['image_url'])}, NULL, NULL, '{r['created_at']}', '{r['updated_at']}');\n")
        f.write('\nGO\nSET IDENTITY_INSERT [lots] OFF;\nGO\n')
        
    with open(FINAL_UNIT_DIR / 'units.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [units] ON;\nGO\n\n')
        for _, r in df_units_final.iterrows():
            f.write(f"INSERT INTO [units] ([id], [number], [legacy_number], [lot_id], [location_id], [status], [condition], [price], [image_url], [vehicle_registration], [burden], [project_id], [created_at], [updated_at], [type], [classification], [vendor_id]) VALUES ({r['id']}, {escape_sql(r['number'])}, {escape_sql(r['legacy_number'])}, {r['lot_id']}, {r['location_id']}, {escape_sql(r['status'])}, {escape_sql(r['condition'])}, {format_num_sql(r['price'])}, {escape_sql(r['image_url'])}, {escape_sql(r['vehicle_registration'])}, {escape_sql(r['burden'])}, {format_num_sql(r['project_id'])}, '{r['created_at']}', '{r['updated_at']}', {escape_sql(r['type'])}, {escape_sql(r['classification'])}, {format_num_sql(r['vendor_id'])});\n")
        f.write('\nGO\nSET IDENTITY_INSERT [units] OFF;\nGO\n')
        
    # Global Quality Assertions
    assert len(df_units_master) == 6943, f'Expected 6,943 units, got {len(df_units_master)}'
    assert len(df_barangs_master) == len(df_lots_master)
    assert (df_barangs_master['id'].astype(int) == list(range(1, len(df_barangs_master) + 1))).all()
    assert (df_lots_master['id'].astype(int) == list(range(1, len(df_lots_master) + 1))).all()
    assert df_barangs_master['number'].is_unique
    assert df_lots_master['number'].is_unique
    assert df_barangs_master['number'].str.len().max() <= 14
    assert df_lots_master['number'].str.len().max() <= 26
    
    print(f' Master Sync Complete: {len(df_barangs_master):,} barangs, {len(df_lots_master):,} lots, {len(df_units_master):,} units.')
    return df_barangs_master, df_lots_master, df_units_master

print('Infrastructure setup and sync engine initialized successfully.')


Infrastructure setup and sync engine initialized successfully.


---
## Subcategory Consolidation: `ACC-AP` (Alat Pemadam Api Ringan / APAR)

### Business Context & Cleaning Rules:
In `ACC-AP`, asset entries originally contained location details typed directly into `spesifikasi` (*"Pos Security Samping"*, *"Depan Resepsionist"*, *"Samping Lift"*), decimal typing variations (`3,5` vs `3.5`), and redundant prefix tags (*"Jenis Apar"*, *"Apar Jenis"*).

**Consolidation Transformations Applied:**
1. **Strip Location Notes**: Physical locations belong in `units.location_id`, not in the catalog definition.
2. **Normalize Decimals**: Unify decimal comma `3,5` $\rightarrow$ `3.5`.
3. **Strip Prefixes**: Remove leading `Jenis Apar`, `Apar Jenis`, `Jenis`.
4. **Standardize Gases**: Standardize `CO 2` $\rightarrow$ `CO2`.
5. **Outcome**: Consolidates **32 fragmented items into exactly 12 standardized catalog barangs** across all 53 units.

In [2]:
# ---------------------------------------------------------
# Subcategory: ACC-AP Consolidation Cell
# ---------------------------------------------------------
SUB_CODE = 'ACC-AP'

def normalize_apar_spec(spec):
    if pd.isna(spec) or not str(spec).strip() or str(spec).strip().lower() in ('nan', 'none', 'null'):
        return None
    s = str(spec).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    
    # 1. Strip leading prefixes
    s = re.sub(r'^(jenis\s+apar\s+|apar\s+jenis\s+|jenis\s+)', '', s, flags=re.IGNORECASE)
    
    # 2. Normalize decimal commas to periods
    s = re.sub(r'(\d+),(\d+)', r'\1.\2', s)
    
    # 3. Standardize CO 2 -> CO2
    s = re.sub(r'co\s*2', 'CO2', s, flags=re.IGNORECASE)
    
    # 4. Extract core type and capacity, stripping location notes
    m_dc = re.search(r'dry\s+chemical\s*(\d+(?:\.\d+)?)\s*(?:kg)?', s, flags=re.IGNORECASE)
    if m_dc:
        return f'Dry Chemical {m_dc.group(1)} kg'
        
    m_co2 = re.search(r'co2\s*(\d+(?:\.\d+)?)\s*(?:kg)?', s, flags=re.IGNORECASE)
    if m_co2:
        return f'CO2 {m_co2.group(1)} kg'
        
    if 'powder' in s.lower() or 'abc' in s.lower():
        m_w = re.search(r'(\d+(?:\.\d+)?)\s*kg', s, flags=re.IGNORECASE)
        w = m_w.group(1) if m_w else '6'
        return f'Powder ABC {w} kg'
        
    return s

# Load partition datasets
df_ap_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_ap_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

df_ap_merged = df_ap_assets.merge(
    df_ap_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

# Group by (brand_id, name, normalized_spec)
groups = defaultdict(list)
for _, r in df_ap_merged.iterrows():
    b_id = resolve_brand_id(r.get('brand'))
    spec_norm = normalize_apar_spec(r.get('spesifikasi'))
    spec_key = spec_norm.upper() if spec_norm else ''
    groups[(b_id, 'APAR', spec_key)].append(r)

# Sort groups deterministically
sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]['id'])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    display_spec = normalize_apar_spec(group_rows[0].get('spesifikasi'))
    
    # First non-null image
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': 'APAR',
        'specification': display_spec,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id in staging
for idx in range(len(df_ap_units)):
    ams_id = str(df_ap_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_ap_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_ap_units.columns:
        df_ap_units[c] = pd.to_numeric(df_ap_units[c], errors='coerce').astype('Int64')
df_ap_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 12, f'Expected 12 barangs, got {len(df_sub_barangs)}'
assert len(df_ap_units) == 53, f'Expected 53 units, got {len(df_ap_units)}'

print(f'\n[SUCCESS] Consolidated ACC-AP from 32 barangs down to {len(df_sub_barangs)} barangs across 53 units.')

# Display the consolidated items table
ap_display = df_sub_barangs[['id', 'number', 'brand_id', 'name', 'specification']].copy()
brand_id_to_name = dict(zip(df_brand['id'], df_brand['name']))
ap_display['brand'] = ap_display['brand_id'].astype(str).map(brand_id_to_name)
ap_unit_counts = df_ap_units.groupby('lot_id').size().to_dict()
ap_display['unit_count'] = ap_display['id'].astype(int).map(ap_unit_counts)
display(ap_display[['id', 'number', 'brand', 'name', 'specification', 'unit_count']])


 Master Sync Complete: 1,470 barangs, 1,470 lots, 6,943 units.

[SUCCESS] Consolidated ACC-AP from 32 barangs down to 12 barangs across 53 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-AP-0001,Appron,APAR,Dry Chemical 3.5 kg,3
1,2,ACC-AP-0002,Firex,APAR,Dry Chemical 3.5 kg,15
2,3,ACC-AP-0003,Appron,APAR,Dry Chemical 2.5 kg,2
3,4,ACC-AP-0004,Firex,APAR,Dry Chemical 2.5 kg,1
4,5,ACC-AP-0005,Firex,APAR,Dry Chemical 3 kg,9
5,6,ACC-AP-0006,Firex,APAR,CO2 7 kg,2
6,7,ACC-AP-0007,Firex,APAR,Dry Chemical 6 kg,3
7,8,ACC-AP-0008,Firex,APAR,CO2 6 kg,2
8,9,ACC-AP-0009,Firex,APAR,CO2 5 kg,2
9,10,ACC-AP-0010,Firex,APAR,CO2 25 kg,1
